# 7 · The value of every ball

On every ball: what the player did, minus what the average player (notebook 4) would have done in the same situation.
Four versions are built (runs, win chance, a skill profile, a mix); notebook 11 chooses between them by testing.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes</summary>

P5 - Performance value per ball (4 candidates), batter and bowler mirror versions.  
V-1 run value above expected   : (runs + RE_next) - E[runs + RE_next | state]   (a wicket costs its situational run value)  
V-2 win-prob added above exp.  : WP_next - E[WP_next | state], de-drifted within WP bins  
V-3 component profile          : reliability-weighted run value of boundary / dot / dismissal / extras residuals  
V-4 hybrid                     : V-1 with the dismissal part shrunk by the dismissal component's reliability  
Batter values use balls faced (no wides); non-striker run-outs are removed from the striker's ledger.  
Bowler values use every delivery and team runs (byes included).
</details>

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from rrlib.common import CFG, load, save, save_table, save_json, get_logger, split_half_reliability

log = get_logger("p05_value")
f = load("features")
c = load("ctx_preds")
p = load("pressure")
for col in c.columns.difference(f.columns):
    f[col] = c[col].values
for col in p.columns.difference(f.columns):
    f[col] = p[col].values

faced = f.batter_faced.values & ~f.nonstriker_out.values
# 16 Nov 2026 review (config ratings.charge_wide_dismissals): a dismissal of the striker off a wide is charged to the
# batter like any other dismissal (before, a wide was 'not a ball faced' and the dismissal was lost)
wdis = (f.is_wide.values & f.striker_out.values.astype(bool)) if CFG["ratings"].get("charge_wide_dismissals", False) else np.zeros(len(f), bool)
rate_wd = wdis.sum() / max(int(f.is_wide.sum()), 1)
yc = f.y_class.values
v = f[["ball_uid", "match_id", "innings", "year", "level", "lvl_code", "striker_id", "bowler_id", "batting_team_id",
       "bowling_team_id", "venue_id", "batting_order", "is_pressure", "pressure", "pressure_pct", "LI", "strain",
       "wp_now", "phase", "is_pace", "knock_out", "comp_id", "match_date", "runs", "team_runs", "is_wicket",
       "striker_out", "legal", "boundary", "is_wide", "is_noball", "bowler_wicket"]].copy()
v["faced"] = faced

#### V-1

In [3]:
v["v1_team"] = f.team_runs + f.re_next - f.E_team_rv
v["v1_bat"] = np.where(faced, f.runs + f.re_next - f.E_bat_rv, np.where(wdis, f.re_next - f.E_bat_rv, np.nan))
v["v1_bowl"] = -v.v1_team

#### V-2 (de-drifted within innings x WP-decile bins)

In [4]:
wpa = f.wp_next - f.E_wp_next
binid = f.innings.astype(str) + "_" + (f.wp_now.clip(0, 0.9999) * 10).astype(int).astype(str)
drift = wpa.groupby(binid).transform("mean")
v["v2_team"] = wpa - drift
v["v2_bat"] = np.where(faced, v.v2_team, np.nan)
v["v2_bowl"] = -v.v2_team
log.info(f"V-2 drift removed: max |bin mean| {wpa.groupby(binid).mean().abs().max():.4f}")

13:40:14 V-2 drift removed: max |bin mean| 0.0014


#### Component residuals (vs M1-base expectation)

In [5]:
pb = np.column_stack([f[c_] for c_ in ["p_dot", "p_1", "p_2", "p_3", "p_4", "p_6", "p_wkt"]])
pb = pb / pb.sum(1, keepdims=True)  # faced-ball (no wide) renormalisation for batters
v["r_bdry_bat"] = np.where(faced, np.isin(yc, [4, 5]) - (pb[:, 4] + pb[:, 5]), np.nan)
v["r_dot_bat"] = np.where(faced, (yc == 0) - pb[:, 0], np.nan)
v["r_wkt_bat"] = np.where(faced, f.striker_out.astype(float) - pb[:, 6], np.where(wdis, 1 - rate_wd, np.nan))
v["r_runs_bat"] = np.where(faced, f.runs - f.x_bat_runs, np.nan)
v["r_bdry_bowl"] = np.isin(yc, [4, 5]) - (f.p_4 + f.p_6)
v["r_dot_bowl"] = (yc == 0) - f.p_dot
# bowler-credited wickets only (run-outs are not the bowler's); expected = P(wicket) x share credited to bowlers in
# that phase (run-outs are more common at the death)
share_bw = (f.bowler_wicket.groupby(f.phase).sum() / f.is_wicket.groupby(f.phase).sum()).reindex(f.phase).values
v["r_wkt_bowl"] = f.bowler_wicket.astype(float) - f.p_wkt * share_bw
v["r_extra_bowl"] = (yc == 7) - f.p_x

#### Run value of each component (regression of V-1 on the residuals) and their reliabilities (all balls)

In [6]:
comps = {"bat": ["r_bdry_bat", "r_dot_bat", "r_wkt_bat"], "bowl": ["r_bdry_bowl", "r_dot_bowl", "r_wkt_bowl", "r_extra_bowl"]}
info = {}
for side, cols in comps.items():
    pcol = "striker_id" if side == "bat" else "bowler_id"
    d = v.dropna(subset=cols + [f"v1_{side}"])
    lr = LinearRegression(fit_intercept=False).fit(d[cols], d[f"v1_{side}"])
    beta = dict(zip(cols, lr.coef_))
    rel = {}
    for cc in cols:
        r, n = split_half_reliability(d, pcol, "match_id", cc, min_n=60)
        rel[cc] = max(0.0, r)
    info[side] = {"beta": beta, "reliability": rel}
    log.info(f"{side}: component run values {({k: round(x, 2) for k, x in beta.items()})}; "
             f"reliabilities {({k: round(x, 2) for k, x in rel.items()})}")
    v[f"v3_{side}"] = sum(rel[cc] * beta[cc] * v[cc] for cc in cols)
    wk = [cc for cc in cols if "wkt" in cc][0]
    v[f"v4_{side}"] = v[f"v1_{side}"] - (1 - rel[wk]) * beta[wk] * v[wk]
b_ = info["bat"]; wk_ = "r_wkt_bat"
v["v3_bat"] = np.where(faced, v.v3_bat, np.where(wdis, b_["reliability"][wk_] * b_["beta"][wk_] * v[wk_], np.nan))
v["v4_bat"] = np.where(faced | wdis, v.v4_bat, np.nan)
log.info(f"dismissals off a wide charged to the batter: {int(wdis.sum())}")

13:40:14 bat: component run values {'r_bdry_bat': np.float64(2.89), 'r_dot_bat': np.float64(-0.89), 'r_wkt_bat': np.float64(-6.24)}; reliabilities {'r_bdry_bat': 0.57, 'r_dot_bat': 0.58, 'r_wkt_bat': 0.22}


13:40:14 bowl: component run values {'r_bdry_bowl': np.float64(-2.93), 'r_dot_bowl': np.float64(0.77), 'r_wkt_bowl': np.float64(6.24), 'r_extra_bowl': np.float64(-1.13)}; reliabilities {'r_bdry_bowl': 0.42, 'r_dot_bowl': 0.32, 'r_wkt_bowl': 0.1, 'r_extra_bowl': 0.58}


13:40:14 dismissals off a wide charged to the batter: 35


#### Reliability table of each value metric (split-half, all vs pressure balls)

In [7]:
rows = []
for side, pcol in (("bat", "striker_id"), ("bowl", "bowler_id")):
    for met in ["v1", "v2", "v3", "v4"]:
        col = f"{met}_{side}"
        d = v.dropna(subset=[col])
        r_all, n_all = split_half_reliability(d, pcol, "match_id", col, min_n=60)
        r_p, n_p = split_half_reliability(d[d.is_pressure], pcol, "match_id", col, min_n=20)
        rows.append({"side": side, "metric": met, "rel_all": r_all, "players_all": n_all, "rel_pressure": r_p,
                     "players_pressure": n_p, "mean": d[col].mean(), "sd": d[col].std()})
R = pd.DataFrame(rows)
save_table(R, "value_metric_reliability")
log.info("\n" + R.round(3).to_string())
save(v, "values")
save_json(info, "p05_components")
log.info("P5 done")

13:40:15 
   side metric  rel_all  players_all  rel_pressure  players_pressure   mean     sd
0   bat     v1    0.216          240         0.116               160  0.008  2.430
1   bat     v2    0.263          240         0.120               164 -0.000  0.030
2   bat     v3    0.426          240         0.290               160  0.002  0.824
3   bat     v4    0.433          240         0.212               160 -0.001  1.962
4  bowl     v1    0.106          270        -0.099               190 -0.032  2.408
5  bowl     v2   -0.152          270         0.252               190  0.000  0.030
6  bowl     v3    0.329          270         0.186               190 -0.001  0.549
7  bowl     v4    0.278          270        -0.038               190 -0.030  1.929


13:40:16 P5 done
